In [ ]:
import random
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from pathlib import Path

import polars as pl
from tqdm.auto import tqdm

from scrapetube.config import (
    QUERY_STRINGS,
    VIDEO_BASE_URL,
    KEYWORD_VIDEOS_META_PARQUET,
    CHANNEL_VIDEOS_META_PARQUET,
    get_today_string,
)
from scrapetube.scrapetube_cc import (
    collect_and_save_video_metadata,
    set_up_logger,
)

today_string = get_today_string()
set_up_logger()

In [ ]:
params = {
    "limit": 20,
    "sleep": (3, 12),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": KEYWORD_VIDEOS_META_PARQUET,
}
query_strings = QUERY_STRINGS[:3]
for q in tqdm(query_strings, desc="Processing keyword videos"):
    params.update({"query": q})
    _ = collect_and_save_video_metadata(**params)

In [ ]:
data_dir = Path("../data")
latest_file = max(data_dir.glob("meta_data*.parquet"), key=lambda f: f.stat().st_mtime)
video_meta_df = pl.read_parquet(latest_file)
unique_channel = [f'"{c}"' for c in video_meta_df["channel"].unique()]
unique_channel

In [ ]:
params = {
    "limit": 5,
    "sleep": (3, 12),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": CHANNEL_VIDEOS_META_PARQUET,
}
for q in tqdm(unique_channel, desc="Processing channel videos"):
    params.update({"query": q})
    _ = collect_and_save_video_metadata(**params)

## Use youtube_transcript_api

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound

In [ ]:
proxies = [
    # {"http": "http://103.25.210.233:9191"},
    # {"http": "http://67.43.227.226:30373"},
    {"http": "http://116.202.121.34:3128"},
    {"http": "http://116.108.3.96:10017"},
    {"http": "http://160.86.242.23:8080"},
    # {"http": "http://35.209.198.222:80"},
    # {"http": "http://20.27.86.185:8080"},
    {"http": "http://31.47.58.37:80"},
    {"http": "http://213.218.255.99:80"},
    {"http": "http://72.10.160.94:8355"},
]

In [ ]:
df = pd.read_csv("./channel_cc_video.csv")
df.info()

In [ ]:
df.sample(5)

In [ ]:
video_id = "mbmckE6eJkQ"
lang_code = ["en"]
available_subtitle = YouTubeTranscriptApi.list_transcripts(video_id)
try:
    subtitles = available_subtitle.find_transcript(lang_code)
    print(subtitles.is_generated)
    print(subtitles.video_id)
except NoTranscriptFound:
    print(f"NO subtitle for lang code {lang_code}")

In [ ]:
# out = YouTubeTranscriptApi.get_transcript(video_id, languages=["th"])
# type(out)
# len(out)

In [ ]:
# video_ids = ["ZP163YC1_9E", "lHsneMSnjgk"]
# try:
#     out = YouTubeTranscriptApi.get_transcripts(video_ids, languages=lang_code)
# except NoTranscriptFound:
#     print("")

In [ ]:
# def get_subtitle_from_video_id(
#         video_id:str,
#         lang_code:list[str],
#         proxies:list[dict[str, str]],
# ):
#     proxy = random.choice(proxies)
#     available_subtitle = YouTubeTranscriptApi.list_transcripts(video_id, proxies=proxy)
#     try:
#         subtitles = available_subtitle.find_transcript(lang_code)
#         return {
#             "video_id" : subtitles.video_id,
#             "is_generate" : subtitles.is_generated,
#             "subtitle" : subtitles.fetch()
#         }
#     except NoTranscriptFound:
#         print(f"NO subtitle for lang code {lang_code}")
#         return None

# lang_code = ["th"]
# subtitle_list = []
# for id in tqdm(df["video_id"], total=len(df), desc="getting subtitle"):
#     subtitle_list.append(
#         get_subtitle_from_video_id(
#             video_id=id,
#             lang_code=lang_code,
#             proxies=proxies,
#         )
#     )
#     sleep(random.randint(3, 10))

In [ ]:
threading.active_count()

In [ ]:
# Function to fetch subtitles
def get_subtitle_from_video_id(
    video_id: str, lang_code: list[str], proxies: list[dict[str, str]]
) -> dict | None:
    """
    Fetch subtitles for a given YouTube video ID.
    """
    proxy = random.choice(proxies)  # Randomly select a proxy
    try:
        available_subtitle = YouTubeTranscriptApi.list_transcripts(
            video_id, proxies=proxy
        )
        subtitles = available_subtitle.find_transcript(lang_code)
        return {
            "video_id": subtitles.video_id,
            "is_generate": subtitles.is_generated,
            "subtitle": subtitles.fetch(),
        }
    except NoTranscriptFound:
        print(f"No subtitle for lang code {lang_code} in video {video_id}")
        print()
        return {
            "video_id": video_id,
            "is_generate": None,
            "subtitle": None,
        }
    except Exception as e:
        print(f"Error fetching subtitles for video {video_id}: {e}")
        print()
        return {
            "video_id": video_id,
            "is_generate": None,
            "subtitle": None,
        }


# Thread-safe function with throttling
def fetch_with_throttle(
    video_id: str,
    lang_code: list[str],
    proxies: list[dict[str, str]],
    sleep_min: int = 3,
    sleep_max: int = 10,
) -> dict | None:
    """
    Wrapper function to introduce throttling between requests.
    """
    result = get_subtitle_from_video_id(video_id, lang_code, proxies)
    sleep(random.randint(sleep_min, sleep_max))  # Random delay to avoid rate limits
    return result


max_workers = 8
lang_code = ["th"]
subtitle_list = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(fetch_with_throttle, video_id, lang_code, proxies)
        for video_id in df["video_id"]
    ]
    for future in tqdm(
        as_completed(futures), total=len(futures), desc="Fetching subtitles"
    ):
        try:
            subtitle_list.append(future.result())
        except Exception as e:
            print(f"Error processing future: {e}")

# Save results to DataFrame
subtitle_df = pd.DataFrame(subtitle_list)

In [ ]:
subtitle_list[0]

In [ ]:
tmp = joblib.dump(
    pd.DataFrame(
        [
            i if i else {"video_id": None, "is_generate": None, "subtitle": None}
            for i in subtitle_list
        ]
    ),
    "tmp.joblib",
)

In [ ]:
tmp_df = pd.DataFrame(
    [
        i if i else {"video_id": None, "is_generate": None, "subtitle": None}
        for i in subtitle_list
    ]
)
# tmp_df = tmp_df.loc[tmp_df["is_generate"].eq(False)]
subs = tmp_df["subtitle"].to_list()

In [ ]:
tmp_df

In [ ]:
len(subs)

In [ ]:
tmp_df.head()

In [ ]:
subs[0][:10]

In [ ]:
text_list = []
end = 0
for sub in subs[:1]:
    text_line = ""
    for s in sub:
        text = s["text"]
        start = s["start"]
        duration = s["duration"]
        if min(start - end, 0) < 1:
            text_line += text
        else:
            text_line += "\n"
            text_line += text

        end = start + duration
    text_list.append(text_line)

In [ ]:
print(text_line)

## tmp